### Load Libraries

In [5]:
from mofapy2.run.entry_point import entry_point
import pandas as pd
import numpy as np

### Loading Data

Load normalized data

In [2]:
mrna_data_lg2 = pd.read_csv("../../data/transformed_data//mrna_data_lg2.csv", index_col=0)
mrna_data_vsn = pd.read_csv("../../data/transformed_data/mrna_data_vsn.csv", index_col=0)
mirna_data_lg2 = pd.read_csv("../../data/transformed_data/mirna_data_lg2.csv", index_col=0)
mirna_data_vsn = pd.read_csv("../../data/transformed_data/mirna_data_vsn.csv", index_col=0)
meth_data = pd.read_csv("../../data/transformed_data/meth_data_m_values.csv", index_col=0)

Load data after feature selection

In [9]:
mrna_data_lg2_fs = pd.read_csv("../../data/feature_selection/selected_features_mrna_data_lg2.csv", index_col=0)
mrna_data_vsn_fs = pd.read_csv("../../data/feature_selection/selected_features_mrna_data_vsn.csv", index_col=0)
mirna_data_lg2_fs = pd.read_csv("../../data/feature_selection/selected_features_mirna_data_lg2.csv", index_col=0)
mirna_data_vsn_fs = pd.read_csv("../../data/feature_selection/selected_features_mirna_data_vsn.csv", index_col=0)
meth_data_fs = pd.read_csv("../../data/feature_selection/selected_features_meth_data.csv", index_col=0)

In [ ]:
runs_config = {"FeatureSelected_LG": {"datasets": [mrna_data_lg2_fs, mirna_data_lg2, meth_data_fs], "save_path": "../../data/latent/mofa_trained_lg2_fs.hdf5"},
                "FeatureSelected_VSN": {"datasets": [mrna_data_vsn_fs, mirna_data_vsn, meth_data_fs], "save_path": "../../data/latent/mofa_trained_vsn_fs.hdf5"},
                "AllFeatures_LG": {"datasets": [mrna_data_lg2, mirna_data_lg2, meth_data], "save_path": "../../data/latent/mofa_trained_lg2.hdf5"},
                "AllFeatures_VSN": {"datasets": [mrna_data_vsn, mirna_data_vsn, meth_data], "save_path": "../../data/latent/mofa_trained_vsn.hdf5"}}

In [11]:
for run_name, config in runs_config.items():

    mrna, mirna, meth = config["datasets"]
    save_path = config["save_path"]

    ent = entry_point()
    data_matrix = [[mrna.values], [mirna.values], [meth.values]]

    views = ["mRNA", "miRNA", "Methylation"]
    samples = [mrna.index.tolist()]
    features = [mrna.columns.tolist(), mirna.columns.tolist(), meth.columns.tolist()]

    ent.set_data_options(scale_groups=False, scale_views=True)

    ent.set_data_matrix(data_matrix, likelihoods=["gaussian", "gaussian", "gaussian"], views_names=views, groups_names = ['PAAD'],samples_names=samples, features_names=features)

    ent.set_model_options(factors = 10, spikeslab_weights = True, ard_weights=True, ard_factors=False)
    ent.set_train_options(iter=1000, convergence_mode="medium", freqELBO=10, gpu_mode=False, seed = 123)

    ent.build()
    ent.run()
    ent.save(save_path)


        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Scaling views to unit variance...

Successfully loaded view='mRNA' group='PAAD' with N=176 samples and D=12819 features...
Successfully loaded view='miRNA' group='PAAD' with N=176 samples and D=298 features...
Successfully loaded view='Methylation' group='PAAD' with N=176 samples and D=60869 features...


Model options:
- Automatic Relevance Determination prior on the factors: False
- 

### Create MOFA data matrix

1 group 3 views

In [4]:
data_matrix_lg2 = [[mrna_data_lg2.values], [mirna_data_lg2.values], [meth_data.values]]

### Define names for samples and features

In [5]:
views = ['mRNA', 'miRNA', 'dna_methylation']
samples_lg2 = [mrna_data_lg2.index.tolist()]
features_lg2 = [mrna_data_lg2.columns.tolist(), mirna_data_lg2.columns.tolist(), meth_data.columns.tolist()]

### Ensure the CpGs do not mathematically overpower the miRNAs

In [6]:
ent.set_data_options(scale_groups=False, scale_views=True)

Scaling views to unit variance...



Sanity Check

In [7]:
print("M (views):", len(data_matrix_lg2))
print("G (groups):", len(data_matrix_lg2[0]))
print("len(views):", len(views))
print("len(samples):", len(samples_lg2))
print("len(features):", len(features_lg2))

M (views): 3
G (groups): 1
len(views): 3
len(samples): 1
len(features): 3


### Load data into the model

since we applied log and M-value transformations, all likelihoods are Gaussian

In [8]:
ent.set_data_matrix(data_matrix_lg2, likelihoods = ['gaussian', 'gaussian', 'gaussian'], views_names = views, groups_names = ['PAAD'], samples_names = samples_lg2, features_names = features_lg2)

Successfully loaded view='mRNA' group='PAAD' with N=176 samples and D=12819 features...
Successfully loaded view='miRNA' group='PAAD' with N=176 samples and D=298 features...
Successfully loaded view='dna_methylation' group='PAAD' with N=176 samples and D=60869 features...




### Set Model Options

- factors: number of factors
- spikeslab_weights: use spike-slab sparsity prior in weights? default is TRUE
- ard_weights: use ARD prior in the weights? Default is TRUE if using multiple views

In [9]:
ent.set_model_options(factors = 10, spikeslab_weights = True, ard_weights = True, ard_factors = False)

Model options:
- Automatic Relevance Determination prior on the factors: False
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (mRNA): gaussian
- View 1 (miRNA): gaussian
- View 2 (dna_methylation): gaussian




### Set Training Options

- convergence_mode: 'fast' (default), 'medium', 'slow'
- dropR2: minimum variance explained criteria to drop factors while training
- gpu_mode: use GPU? (needs cupy installed and a functional GPU, see https://biofam.github.io/MOFA2/gpu_training.html)
- seed: random seed

In [10]:
ent.set_train_options(iter = 1000, convergence_mode = 'medium', freqELBO = 10, gpu_mode = False, seed = 123)

### Build and Train the MOFA model

In [11]:
ent.build()
ent.run()

ent.save('../../data/latent/mofa_trained_lg2_fs.hdf5')



######################################
## Training the model with seed 123 ##
######################################


ELBO before training: -101426695.78 

Iteration 1: time=3.69, ELBO=-15990184.13, deltaELBO=85436511.646 (84.23473819%), Factors=10
Iteration 2: time=3.40, Factors=10
Iteration 3: time=3.36, Factors=10
Iteration 4: time=3.18, Factors=10
Iteration 5: time=3.52, Factors=10
Iteration 6: time=4.78, Factors=10
Iteration 7: time=6.21, Factors=10
Iteration 8: time=4.73, Factors=10
Iteration 9: time=4.63, Factors=10
Iteration 10: time=3.92, Factors=10
Iteration 11: time=4.05, ELBO=-12481992.50, deltaELBO=3508191.630 (3.45884444%), Factors=10
Iteration 12: time=4.12, Factors=10
Iteration 13: time=3.35, Factors=10
Iteration 14: time=3.22, Factors=10
Iteration 15: time=3.11, Factors=10
Iteration 16: time=3.48, Factors=10
Iteration 17: time=3.24, Factors=10
Iteration 18: time=3.02, Factors=10
Iteration 19: time=3.28, Factors=10
Iteration 20: time=3.45, Factors=10
Iteration 21: ti